# Short-horizon rebuild notebook (1h, 6h, 12h, 24h)

This notebook rebuilds the eight-model comparison from the original project using the same overall methods, but with shorter forecast horizons.

The model set is:

1. Elastic Net  
2. Random Forest  
3. LSTM  
4. CNN–GRU–LSTM  
5. GraphWaveNet  
6. GraphWaveNet–GRU  
7. GraphWaveNet–LSTM  
8. GraphWaveNet–GRU–LSTM  

The methodological goal is to keep the project aligned with the original notebooks:

- same split dates
- same station set
- same feature channels
- same train-only scaling
- same strict sliding-window logic
- same evaluation style in original traffic-flow units

The only forecasting change is:

- old setup: `OUT_LEN = 72`, report `12, 24, 48, 72`
- new setup: `OUT_LEN = 24`, report `1, 6, 12, 24`


In [1]:
from pathlib import Path
import copy
import gc
import json
import random
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.ensemble import RandomForestRegressor
from joblib import Parallel, delayed

from IPython.display import display


def set_seed(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


SEED = 42
set_seed(SEED, deterministic=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# Raw files
# -------------------------
TRAFFIC_CSV = Path("cleaned_traffic_data.csv")
META_XLSX   = Path("pems_output.xlsx")

# -------------------------
# Existing strict artifact from the original project
# -------------------------
BASE_STRICT_DATASET = Path("artifacts/pems_graph_dataset_strict.npz")

# -------------------------
# Split boundaries (same as original project)
# -------------------------
TRAIN_END = pd.Timestamp("2024-11-15 23:59:59")
VAL_END   = pd.Timestamp("2024-11-30 23:59:59")

# -------------------------
# Short-horizon setup
# -------------------------
IN_LEN = 24
OUT_LEN = 24
EVAL_HORIZONS = [1, 6, 12, 24]

assert max(EVAL_HORIZONS) <= OUT_LEN, "The largest evaluation horizon must be <= OUT_LEN."

SHORT_TAG = "short_h1_6_12_24"

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHORT_DATASET = OUT_DIR / f"pems_graph_dataset_strict_{SHORT_TAG}.npz"
RESULTS_SUMMARY_SHORT = OUT_DIR / f"results_summary_{SHORT_TAG}.csv"

print("BASE_STRICT_DATASET:", BASE_STRICT_DATASET)
print("SHORT_DATASET:", SHORT_DATASET)
print("RESULTS_SUMMARY_SHORT:", RESULTS_SUMMARY_SHORT)

Torch: 2.1.1+cu121
Device: cpu
BASE_STRICT_DATASET: artifacts/pems_graph_dataset_strict.npz
SHORT_DATASET: artifacts/pems_graph_dataset_strict_short_h1_6_12_24.npz
RESULTS_SUMMARY_SHORT: artifacts/results_summary_short_h1_6_12_24.csv


## Rebuild the strict dataset for 24-step forecasting

This is the key short-horizon conversion step.

To stay methodologically faithful to the original project, we do not rebuild the whole cleaned dataset from scratch here.  
Instead, we:

- load the already-built strict artifact from the original notebook
- keep the same `X`, `Y`, adjacency, stations, timestamps, and train-only statistics
- recompute only the valid start indices and split assignments for `OUT_LEN = 24`

That way, the study changes only in the forecast length and evaluation horizons.

In [2]:
assert BASE_STRICT_DATASET.exists(), (
    f"Missing {BASE_STRICT_DATASET}. "
    "Run the original notebook once so the strict base artifact exists."
)

base = np.load(BASE_STRICT_DATASET, allow_pickle=True)

X_base = base["X"].astype(np.float32)
Y_base = base["Y"].astype(np.float32)
A_base = base["A"].astype(np.float32)
stations_base = base["stations"]
timestamps_base = pd.to_datetime(base["timestamps"])

flow_mean_base = base["flow_mean"].astype(np.float32)
flow_std_base  = base["flow_std"].astype(np.float32)
speed_mean_base = base["speed_mean"].astype(np.float32)
speed_std_base  = base["speed_std"].astype(np.float32)

base_in_len = int(np.array(base["in_len"]).item())
print("Base strict artifact loaded.")
print("Base X:", X_base.shape, "Base Y:", Y_base.shape)
print("Base IN_LEN:", base_in_len)

assert base_in_len == IN_LEN, (
    f"Expected base IN_LEN={IN_LEN}, found {base_in_len}. "
    "This notebook assumes the original project used 24 hours of input."
)

T_total = X_base.shape[0]
max_start = T_total - (IN_LEN + OUT_LEN) + 1
starts = np.arange(max_start, dtype=np.int32)

out_start_times = timestamps_base[starts + IN_LEN]
out_end_times   = timestamps_base[starts + IN_LEN + OUT_LEN - 1]

train_starts_short = starts[out_end_times <= TRAIN_END]
val_starts_short   = starts[(out_start_times > TRAIN_END) & (out_end_times <= VAL_END)]
test_starts_short  = starts[out_start_times > VAL_END]

print("Short-horizon strict window counts:")
print("train:", len(train_starts_short))
print("val:  ", len(val_starts_short))
print("test: ", len(test_starts_short))

np.savez_compressed(
    SHORT_DATASET,
    X=X_base.astype(np.float32),
    Y=Y_base.astype(np.float32),
    A=A_base.astype(np.float32),
    stations=stations_base,
    timestamps=np.array(timestamps_base.astype("datetime64[ns]")),
    train_starts=train_starts_short,
    val_starts=val_starts_short,
    test_starts=test_starts_short,
    in_len=np.array([IN_LEN], dtype=np.int32),
    out_len=np.array([OUT_LEN], dtype=np.int32),
    flow_mean=flow_mean_base,
    flow_std=flow_std_base,
    speed_mean=speed_mean_base,
    speed_std=speed_std_base,
)

print("Saved short-horizon strict dataset to:", SHORT_DATASET)

Base strict artifact loaded.
Base X: (2208, 1821, 6) Base Y: (2208, 1821)
Base IN_LEN: 24
Short-horizon strict window counts:
train: 1057
val:   337
test:  721
Saved short-horizon strict dataset to: artifacts/pems_graph_dataset_strict_short_h1_6_12_24.npz


## Load the short-horizon shared tensors

We keep the same shared representation used in the original notebooks:

- `X` with shape `(T, N, F)`
- `Y` with shape `(T, N)`

The feature channels remain:

1. flow  
2. speed  
3. hour_sin  
4. hour_cos  
5. dow_sin  
6. dow_cos  

Only the output sequence length changes from `72` to `24`.

In [3]:
assert SHORT_DATASET.exists(), f"Missing {SHORT_DATASET}. Run the rebuild cell above first."

data = np.load(SHORT_DATASET, allow_pickle=True)

X = data["X"].astype(np.float32)          # (T, N, F)
Y = data["Y"].astype(np.float32)          # (T, N)
A = data["A"].astype(np.float32)          # (N, N)

stations = data["stations"]
timestamps = pd.to_datetime(data["timestamps"])

train_starts = data["train_starts"].astype(np.int64)
val_starts   = data["val_starts"].astype(np.int64)
test_starts  = data["test_starts"].astype(np.int64)

IN_LEN  = int(np.array(data["in_len"]).item())
OUT_LEN = int(np.array(data["out_len"]).item())

flow_mean  = data["flow_mean"].astype(np.float32)
flow_std   = data["flow_std"].astype(np.float32)
speed_mean = data["speed_mean"].astype(np.float32)
speed_std  = data["speed_std"].astype(np.float32)

flow_std  = np.maximum(flow_std, 1e-6).astype(np.float32)
speed_std = np.maximum(speed_std, 1e-6).astype(np.float32)

T, N, Fdim = X.shape
print("X:", X.shape, "(T,N,F)")
print("Y:", Y.shape, "(T,N)")
print("A:", A.shape)
print("IN_LEN / OUT_LEN:", IN_LEN, OUT_LEN)
print("Stations:", N)
print("train / val / test starts:", len(train_starts), len(val_starts), len(test_starts))


def time_encoding(dt_index: pd.DatetimeIndex) -> np.ndarray:
    hours = dt_index.hour.values
    dow   = dt_index.dayofweek.values
    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    dow_sin  = np.sin(2 * np.pi * dow / 7.0)
    dow_cos  = np.cos(2 * np.pi * dow / 7.0)
    return np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1).astype(np.float32)


TF_all = time_encoding(pd.DatetimeIndex(timestamps))  # (T, 4)

# Scale only the continuous sensor channels using train-only stats saved in the artifact.
X_scaled = X.copy()
X_scaled[:, :, 0] = (X_scaled[:, :, 0] - flow_mean[None, :]) / flow_std[None, :]
X_scaled[:, :, 1] = (X_scaled[:, :, 1] - speed_mean[None, :]) / speed_std[None, :]

Y_scaled = (Y - flow_mean[None, :]) / flow_std[None, :]

# Conv-style layout reused in the original notebooks: (F, N, T)
X_fnt = np.transpose(X_scaled, (2, 1, 0)).copy()

print("TF_all:", TF_all.shape)
print("X_fnt:", X_fnt.shape)
print("Scaled target mean/std:", float(Y_scaled.mean()), float(Y_scaled.std()))

X: (2208, 1821, 6) (T,N,F)
Y: (2208, 1821) (T,N)
A: (1821, 1821)
IN_LEN / OUT_LEN: 24 24
Stations: 1821
train / val / test starts: 1057 337 721
TF_all: (2208, 4)
X_fnt: (6, 1821, 2208)
Scaled target mean/std: -781.403564453125 30704.76953125


## Shared dataset and dataloaders for all deep models

All deep models use the same sliding-window dataset:

- `x`: `(F, N, IN_LEN)`
- `y`: `(OUT_LEN, N)` in scaled flow units
- `tf`: `(OUT_LEN, 4)` future calendar features for the target steps

This matches the later clean sections of the original project.

In [4]:
class FastPemsWindowDataset(Dataset):
    def __init__(self, X_fnt, Y_scaled, TF_all, starts, in_len, out_len):
        self.X_fnt = X_fnt
        self.Y = Y_scaled
        self.TF = TF_all
        self.starts = np.asarray(starts, dtype=np.int64)
        self.in_len = int(in_len)
        self.out_len = int(out_len)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = int(self.starts[idx])
        x = self.X_fnt[:, :, s:s + self.in_len]                    # (F, N, IN)
        y = self.Y[s + self.in_len:s + self.in_len + self.out_len, :]  # (OUT, N)
        tf = self.TF[s + self.in_len:s + self.in_len + self.out_len, :] # (OUT, 4)

        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(y).float(),
            torch.from_numpy(tf).float(),
        )


train_ds = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, train_starts, IN_LEN, OUT_LEN)
val_ds   = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, val_starts,   IN_LEN, OUT_LEN)
test_ds  = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, test_starts,  IN_LEN, OUT_LEN)

BATCH_SIZE = 8

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

xb, yb, tfb = next(iter(train_loader))
print("Batch x:", xb.shape)
print("Batch y:", yb.shape)
print("Batch tf:", tfb.shape)

Batch x: torch.Size([8, 6, 1821, 24])
Batch y: torch.Size([8, 24, 1821])
Batch tf: torch.Size([8, 24, 4])
